### Locality analysis

To better evaluate the minimal-change property, we additionally measure the locality of the newly added patch faces. Specifically, we define a repair zone by the removed-part bounding box with a small margin, and classify each added face according to whether its face center lies inside or outside this zone.

On the test set, both baselines remain largely local under this metric:

- **Center-fan baseline**
  - mean added faces inside zone: 32.3
  - mean added faces outside zone: 14.6
  - mean face locality ratio: 0.6786

- **Planar triangulation baseline**
  - mean added faces inside zone: 28.8
  - mean added faces outside zone: 8.2
  - mean face locality ratio: 0.6744

Overall, the two baselines show very similar locality ratios. However, the planar triangulation baseline uses fewer added faces in total and introduces fewer faces outside the repair zone, suggesting a slightly cleaner local repair behavior.

In [8]:
import json
import numpy as np
import open3d as o3d
from pathlib import Path
from collections import defaultdict
from scipy.spatial import Delaunay
from matplotlib.path import Path as MplPath
from baseline_center_fan import repair_one_sample_center_fan
from baseline_planar_triangulation import repair_one_sample_planar
# ========= 改成你的 test 集 sample 目录列表 =========
dataset_root = Path("./dataset")

with open(dataset_root / "split.json", "r", encoding="utf-8") as f:
    split = json.load(f)

test_ids = split["test"]
test_sample_dirs = [dataset_root / sid for sid in test_ids]

# 如果你已经有固定 test id 列表，更建议直接写：
# test_ids = ["42103", "35580", ...]
# test_sample_dirs = [dataset_root / sid for sid in test_ids]


# ========= 这里假设你已经分别定义好了两个 repair_one_sample =========
# 建议你先把两个 baseline 的 repair_one_sample 重命名：
# repair_one_sample_center_fan(sample_dir)
# repair_one_sample_planar(sample_dir)


def run_and_collect(sample_dirs, repair_fn):
    rows = []
    for sample_dir in sample_dirs:
        try:
            out = repair_fn(sample_dir)

            row = {
                "sample_id": out.get("sample_id"),
                "success": out.get("success", False),
                "reason": out.get("reason", ""),
                "num_loops_repaired": out.get("num_loops_repaired", 0),
                "total_loop_len_before": out.get("total_loop_len_before", 0.0),
                "nearest_loop_len_after": out.get("nearest_loop_len_after", 0.0),
                "improvement": out.get("improvement", 0.0),
                "num_new_vertices": out.get("num_new_vertices", 0),
                "num_added_faces": out.get("num_added_faces", 0),
                "mean_added_face_quality": out.get("mean_added_face_quality", 0.0),
                "min_added_face_quality": out.get("min_added_face_quality", 0.0),

                # locality
                "num_added_faces_inside_zone": out.get("num_added_faces_inside_zone", 0),
                "num_added_faces_outside_zone": out.get("num_added_faces_outside_zone", 0),
                "face_locality_ratio": out.get("face_locality_ratio", 0.0),
            }
            rows.append(row)

        except Exception as e:
            rows.append({
                "sample_id": Path(sample_dir).name,
                "success": False,
                "reason": f"exception: {e}",
                "num_loops_repaired": 0,
                "total_loop_len_before": 0.0,
                "nearest_loop_len_after": 0.0,
                "improvement": 0.0,
                "num_new_vertices": 0,
                "num_added_faces": 0,
                "mean_added_face_quality": 0.0,
                "min_added_face_quality": 0.0,
                "num_added_faces_inside_zone": 0,
                "num_added_faces_outside_zone": 0,
                "face_locality_ratio": 0.0,
            })

    return pd.DataFrame(rows)


def build_summary(df):
    df_ok = df[df["success"] == True].copy()

    summary = pd.DataFrame([{
        "num_samples": len(df),
        "num_success": int(df["success"].sum()),
        "mean_total_loop_len_before": df_ok["total_loop_len_before"].mean(),
        "mean_nearest_loop_len_after": df_ok["nearest_loop_len_after"].mean(),
        "mean_improvement": df_ok["improvement"].mean(),
        "mean_num_loops_repaired": df_ok["num_loops_repaired"].mean(),
        "mean_num_new_vertices": df_ok["num_new_vertices"].mean(),
        "mean_num_added_faces": df_ok["num_added_faces"].mean(),
        "mean_added_face_quality": df_ok["mean_added_face_quality"].mean(),
        "mean_min_added_face_quality": df_ok["min_added_face_quality"].mean(),

        # locality summary
        "mean_num_added_faces_inside_zone": df_ok["num_added_faces_inside_zone"].mean(),
        "mean_num_added_faces_outside_zone": df_ok["num_added_faces_outside_zone"].mean(),
        "mean_face_locality_ratio": df_ok["face_locality_ratio"].mean(),
    }])

    return summary


In [9]:
# ========= 跑 center-fan =========
df_center = run_and_collect(test_sample_dirs, repair_one_sample_center_fan)
summary_center = build_summary(df_center)

df_center.to_csv("baseline_center_fan_results_locality.csv", index=False)
summary_center.to_csv("baseline_center_fan_summary_locality.csv", index=False)

print("Center-fan done")
display(df_center.head())
display(summary_center)

Center-fan done


,sample_id,success,reason,num_loops_repaired,total_loop_len_before,nearest_loop_len_after,improvement,num_new_vertices,num_added_faces,mean_added_face_quality,min_added_face_quality,num_added_faces_inside_zone,num_added_faces_outside_zone,face_locality_ratio
0,35580,True,,4,1.172208,0.0,1.172208,4,59,0.634401,0.083625,59,0,1.0
1,42103,True,,2,1.476778,0.0,1.476778,2,48,0.437962,0.189188,48,0,1.0
2,35871,True,,2,1.461360,0.0,1.461360,2,48,0.443731,0.182823,48,0,1.0
3,36192,True,no boundary loops found,0,0.000000,0.0,0.000000,0,0,0.000000,0.000000,0,0,0.0
4,37675,True,,3,0.571676,0.0,0.571676,3,37,0.748533,0.412540,37,0,1.0


,num_samples,num_success,mean_total_loop_len_before,mean_nearest_loop_len_after,mean_improvement,mean_num_loops_repaired,mean_num_new_vertices,mean_num_added_faces,mean_added_face_quality,mean_min_added_face_quality,mean_num_added_faces_inside_zone,mean_num_added_faces_outside_zone,mean_face_locality_ratio
0,10,10,0.668064,0.0,0.668064,1.8,1.8,46.9,0.401098,0.190122,32.3,14.6,0.678571


In [10]:
# ========= 跑 planar triangulation =========
df_planar = run_and_collect(test_sample_dirs, repair_one_sample_planar)
summary_planar = build_summary(df_planar)

df_planar.to_csv("baseline_planar_results_locality.csv", index=False)
summary_planar.to_csv("baseline_planar_summary_locality.csv", index=False)

print("Planar triangulation done")
display(df_planar.head())
display(summary_planar)

Planar triangulation done


,sample_id,success,reason,num_loops_repaired,total_loop_len_before,nearest_loop_len_after,improvement,num_new_vertices,num_added_faces,mean_added_face_quality,min_added_face_quality,num_added_faces_inside_zone,num_added_faces_outside_zone,face_locality_ratio
0,35580,True,,4,1.172208,0.0,1.172208,0,50,0.617853,0.310262,50,0,1.0
1,42103,True,,2,1.476778,0.0,1.476778,0,44,0.454382,0.266545,44,0,1.0
2,35871,True,,2,1.461360,0.0,1.461360,0,44,0.456530,0.278732,44,0,1.0
3,36192,True,no boundary loops found,0,0.000000,0.0,0.000000,0,0,0.000000,0.000000,0,0,0.0
4,37675,True,,3,0.571676,0.0,0.571676,0,31,0.608512,0.377951,31,0,1.0


,num_samples,num_success,mean_total_loop_len_before,mean_nearest_loop_len_after,mean_improvement,mean_num_loops_repaired,mean_num_new_vertices,mean_num_added_faces,mean_added_face_quality,mean_min_added_face_quality,mean_num_added_faces_inside_zone,mean_num_added_faces_outside_zone,mean_face_locality_ratio
0,10,10,0.668064,0.0,0.668064,1.8,0.0,37.0,0.411637,0.223834,28.8,8.2,0.674359


In [11]:
pd.set_option("display.max_columns", None)

print(summary_center)
print(summary_planar)

   num_samples  num_success  mean_total_loop_len_before  \
0           10           10                    0.668064   

   mean_nearest_loop_len_after  mean_improvement  mean_num_loops_repaired  \
0                          0.0          0.668064                      1.8   

   mean_num_new_vertices  mean_num_added_faces  mean_added_face_quality  \
0                    1.8                  46.9                 0.401098   

   mean_min_added_face_quality  mean_num_added_faces_inside_zone  \
0                     0.190122                              32.3   

   mean_num_added_faces_outside_zone  mean_face_locality_ratio  
0                               14.6                  0.678571  
   num_samples  num_success  mean_total_loop_len_before  \
0           10           10                    0.668064   

   mean_nearest_loop_len_after  mean_improvement  mean_num_loops_repaired  \
0                          0.0          0.668064                      1.8   

   mean_num_new_vertices  mean_num